In [1]:
import sympy as sp

# ============================================================
# Coordinates
# ============================================================

t, r, th, ph = sp.symbols("t r theta phi", real=True)
coords = [t, r, th, ph]

# ============================================================
# Functions
# ============================================================

a = sp.Function("a")(t)
c = sp.Function("c")(t)
b = sp.Function("b")(t, r)

adot = sp.diff(a, t)
br = sp.diff(b, r)

X = sp.simplify(adot - b)

# ============================================================
# Tetrad e^a_mu
# ============================================================

eta = sp.diag(-1, 1, 1, 1)

e = sp.zeros(4, 4)

e[0, 0] = c
e[1, 0] = r*b
e[1, 1] = a
e[2, 2] = r*a
e[3, 3] = r*a*sp.sin(th)

e_inv = sp.simplify(e.inv())

# ============================================================
# Metric
# ============================================================

g = sp.simplify(e.T * eta * e)
g_inv = sp.simplify(g.inv())

# ============================================================
# Spin connection omega^a_{b mu}
# ============================================================

omega = [[[
    sp.Integer(0)
    for mu in range(4)]
    for B in range(4)]
    for A in range(4)
]

omega[1][2][2] = -1
omega[2][1][2] = 1

omega[1][3][3] = -sp.sin(th)
omega[3][1][3] = sp.sin(th)

omega[2][3][3] = -sp.cos(th)
omega[3][2][3] = sp.cos(th)

# ============================================================
# Lorentz torsion T^a_{mu nu}
# ============================================================

T_lor = [[[
    sp.simplify(
        sp.diff(e[A, nu], coords[mu])
        - sp.diff(e[A, mu], coords[nu])
        + sum(omega[A][B][mu] * e[B, nu] for B in range(4))
        - sum(omega[A][B][nu] * e[B, mu] for B in range(4))
    )
    for nu in range(4)]
    for mu in range(4)]
    for A in range(4)
]

# ============================================================
# Coordinate torsion T^rho_{mu nu}
# ============================================================

T_up_down_down = [[[
    sp.simplify(
        sum(e_inv[rho, A] * T_lor[A][mu][nu] for A in range(4))
    )
    for nu in range(4)]
    for mu in range(4)]
    for rho in range(4)
]

# ============================================================
# Fully lowered torsion T_{rho mu nu}
# ============================================================

T_down_down_down = [[[
    sp.simplify(
        sum(g[rho, sig] * T_up_down_down[sig][mu][nu] for sig in range(4))
    )
    for nu in range(4)]
    for mu in range(4)]
    for rho in range(4)
]

# ============================================================
# Fully raised torsion T^{rho mu nu}
# ============================================================

T_up_up_up = [[[
    sp.simplify(
        sum(
            g_inv[rho, sig]
            * g_inv[mu, alpha]
            * g_inv[nu, beta]
            * T_down_down_down[sig][alpha][beta]
            for sig in range(4)
            for alpha in range(4)
            for beta in range(4)
        )
    )
    for nu in range(4)]
    for mu in range(4)]
    for rho in range(4)
]

# ============================================================
# Torsion vector V_mu = T^rho_{rho mu}
# ============================================================

V_cov = [
    sp.simplify(
        sum(T_up_down_down[rho][rho][mu] for rho in range(4))
    )
    for mu in range(4)
]

V_contra = [
    sp.simplify(
        sum(g_inv[mu, nu] * V_cov[nu] for nu in range(4))
    )
    for mu in range(4)
]

# ============================================================
# Torsion scalar
#
# T = 1/4 T^{rho mu nu} T_{rho mu nu}
#   + 1/2 T^{rho mu nu} T_{nu mu rho}
#   - T^rho_{rho mu} T^nu_{nu}^{ mu}
# ============================================================

term1 = sp.simplify(
    sp.Rational(1, 4)
    * sum(
        T_up_up_up[rho][mu][nu]
        * T_down_down_down[rho][mu][nu]
        for rho in range(4)
        for mu in range(4)
        for nu in range(4)
    )
)

term2 = sp.simplify(
    sp.Rational(1, 2)
    * sum(
        T_up_up_up[rho][mu][nu]
        * T_down_down_down[nu][mu][rho]
        for rho in range(4)
        for mu in range(4)
        for nu in range(4)
    )
)

term3 = sp.simplify(
    -sum(V_cov[mu] * V_contra[mu] for mu in range(4))
)

Tscalar = sp.simplify(sp.factor(term1 + term2 + term3))

# ============================================================
# Expected result
# ============================================================

T_expected = sp.simplify(
    2 * X * (3*X - 2*r*br) / (a**2 * c**2)
)

check = sp.simplify(sp.factor(Tscalar - T_expected))

# ============================================================
# FLRW limit
# ============================================================

T_FLRW = sp.simplify(
    Tscalar.subs({
        b: 0,
        br: 0,
        c: 1
    })
)

# ============================================================
# Print
# ============================================================

print("\nTetrad e^a_mu:")
sp.pprint(e)

print("\nInverse tetrad e_a^mu:")
sp.pprint(e_inv)

print("\nMetric g_mu_nu:")
sp.pprint(g)

print("\nNonzero Lorentz torsion T^a_{mu nu}:")
for A in range(4):
    for mu in range(4):
        for nu in range(4):
            expr = sp.simplify(T_lor[A][mu][nu])
            if expr != 0:
                print(f"T^{A}_{{{mu}{nu}}} =")
                sp.pprint(sp.factor(expr))
                print()

print("\nNonzero coordinate torsion T^rho_{mu nu}:")
for rho in range(4):
    for mu in range(4):
        for nu in range(4):
            expr = sp.simplify(T_up_down_down[rho][mu][nu])
            if expr != 0:
                print(f"T^{rho}_{{{mu}{nu}}} =")
                sp.pprint(sp.factor(expr))
                print()

print("\nTorsion vector V_mu = T^rho_{rho mu}:")
for mu in range(4):
    print(f"V_{mu} =")
    sp.pprint(sp.factor(V_cov[mu]))
    print()

print("\nterm1 =")
sp.pprint(sp.factor(term1))

print("\nterm2 =")
sp.pprint(sp.factor(term2))

print("\nterm3 =")
sp.pprint(sp.factor(term3))

print("\nTorsion scalar T =")
sp.pprint(sp.factor(Tscalar))

print("\nExpected T =")
sp.pprint(sp.factor(T_expected))

print("\nCheck T - Expected =")
sp.pprint(check)

print("\nFLRW limit b=0, c=1:")
sp.pprint(sp.factor(T_FLRW))


Tetrad e^a_mu:
⎡  c(t)      0      0           0      ⎤
⎢                                      ⎥
⎢r⋅b(t, r)  a(t)    0           0      ⎥
⎢                                      ⎥
⎢    0       0    r⋅a(t)        0      ⎥
⎢                                      ⎥
⎣    0       0      0     r⋅a(t)⋅sin(θ)⎦

Inverse tetrad e_a^mu:
⎡    1                                   ⎤
⎢   ────       0      0           0      ⎥
⎢   c(t)                                 ⎥
⎢                                        ⎥
⎢-r⋅b(t, r)    1                         ⎥
⎢───────────  ────    0           0      ⎥
⎢ a(t)⋅c(t)   a(t)                       ⎥
⎢                                        ⎥
⎢                     1                  ⎥
⎢     0        0    ──────        0      ⎥
⎢                   r⋅a(t)               ⎥
⎢                                        ⎥
⎢                                 1      ⎥
⎢     0        0      0     ─────────────⎥
⎣                           r⋅a(t)⋅sin(θ)⎦

Metric g_mu_nu:
⎡ 2  2     

In [2]:
# ============================================================
# T^{mu nu}_{ rho}
# ============================================================

T_up_up_down = [[[
    sp.simplify(
        sum(
            g_inv[mu, alpha]
            * g_inv[nu, beta]
            * T_down_down_down[alpha][beta][rho]
            for alpha in range(4)
            for beta in range(4)
        )
    )
    for rho in range(4)]
    for nu in range(4)]
    for mu in range(4)
]

# ============================================================
# T_{rho}^{ mu nu}
# ============================================================

T_down_up_up = [[[
    sp.simplify(
        sum(
            g_inv[mu, alpha]
            * g_inv[nu, beta]
            * T_down_down_down[rho][alpha][beta]
            for alpha in range(4)
            for beta in range(4)
        )
    )
    for nu in range(4)]
    for mu in range(4)]
    for rho in range(4)
]

# ============================================================
# Kontorsiyon K^{mu nu}_{ rho}
#
# K^{mu nu}_{ rho}
# =
# -1/2 ( T^{mu nu}_{ rho}
#       -T^{nu mu}_{ rho}
#       -T_{rho}^{ mu nu} )
# ============================================================

K_up_up_down = [[[
    sp.simplify(
        -sp.Rational(1, 2) *
        (
            T_up_up_down[mu][nu][rho]
            - T_up_up_down[nu][mu][rho]
            - T_down_up_up[rho][mu][nu]
        )
    )
    for rho in range(4)]
    for nu in range(4)]
    for mu in range(4)
]

# ============================================================
# Trace Q^nu = T^{alpha nu}_{ alpha}
# ============================================================

Q_contra = [
    sp.simplify(
        sum(T_up_up_down[alpha][nu][alpha] for alpha in range(4))
    )
    for nu in range(4)
]

# ============================================================
# Superpotential S_rho^{mu nu}
#
# S_rho^{mu nu}
# =
# 1/2 ( K^{mu nu}_{ rho}
#      +delta^mu_rho T^{alpha nu}_{ alpha}
#      -delta^nu_rho T^{alpha mu}_{ alpha} )
# ============================================================

S_down_up_up = [[[
    sp.simplify(
        sp.Rational(1, 2) *
        (
            K_up_up_down[mu][nu][rho]
            + (1 if mu == rho else 0) * Q_contra[nu]
            - (1 if nu == rho else 0) * Q_contra[mu]
        )
    )
    for nu in range(4)]
    for mu in range(4)]
    for rho in range(4)
]

# ============================================================
# Check: T = S_rho^{mu nu} T^rho_{mu nu}
# ============================================================

T_from_S = sp.simplify(
    sp.factor(
        sum(
            S_down_up_up[rho][mu][nu] * T_up_down_down[rho][mu][nu]
            for rho in range(4)
            for mu in range(4)
            for nu in range(4)
        )
    )
)

check_S = sp.simplify(sp.factor(T_from_S - Tscalar))

# ============================================================
# Print nonzero kontorsiyon
# ============================================================

print("\n============================================================")
print("Nonzero contorsion K^{mu nu}_{rho}")
print("============================================================")

for mu in range(4):
    for nu in range(4):
        for rho in range(4):
            expr = sp.simplify(K_up_up_down[mu][nu][rho])
            if expr != 0:
                print(f"K^{{{mu}{nu}}}_{{{rho}}} =")
                sp.pprint(sp.factor(expr))
                print()

# ============================================================
# Print Q^nu
# ============================================================

print("\n============================================================")
print("Trace Q^nu = T^{alpha nu}_{alpha}")
print("============================================================")

for nu in range(4):
    print(f"Q^{nu} =")
    sp.pprint(sp.factor(Q_contra[nu]))
    print()

# ============================================================
# Print nonzero superpotential
# ============================================================

print("\n============================================================")
print("Nonzero superpotential S_rho^{mu nu}")
print("============================================================")

for rho in range(4):
    for mu in range(4):
        for nu in range(4):
            expr = sp.simplify(S_down_up_up[rho][mu][nu])
            if expr != 0:
                print(f"S_{rho}^{{{mu}{nu}}} =")
                sp.pprint(sp.factor(expr))
                print()

# ============================================================
# Final checks
# ============================================================

print("\n============================================================")
print("T from superpotential")
print("============================================================")
sp.pprint(sp.factor(T_from_S))

print("\n============================================================")
print("Previous Tscalar")
print("============================================================")
sp.pprint(sp.factor(Tscalar))

print("\n============================================================")
print("Check T_from_S - Tscalar")
print("============================================================")
sp.pprint(check_S)


Nonzero contorsion K^{mu nu}_{rho}
K^{01}_{0} =
  ⎛  ∂                       d       ⎞        
r⋅⎜r⋅──(b(t, r)) + b(t, r) - ──(a(t))⎟⋅b(t, r)
  ⎝  ∂r                      dt      ⎠        
──────────────────────────────────────────────
                  2     2                     
                 a (t)⋅c (t)                  

K^{01}_{1} =
  ∂                       d       
r⋅──(b(t, r)) + b(t, r) - ──(a(t))
  ∂r                      dt      
──────────────────────────────────
                  2               
            a(t)⋅c (t)            

K^{02}_{2} =
 ⎛           d       ⎞ 
-⎜-b(t, r) + ──(a(t))⎟ 
 ⎝           dt      ⎠ 
───────────────────────
            2          
      a(t)⋅c (t)       

K^{03}_{3} =
 ⎛           d       ⎞ 
-⎜-b(t, r) + ──(a(t))⎟ 
 ⎝           dt      ⎠ 
───────────────────────
            2          
      a(t)⋅c (t)       

K^{10}_{0} =
   ⎛  ∂                       d       ⎞         
-r⋅⎜r⋅──(b(t, r)) + b(t, r) - ──(a(t))⎟⋅b(t, r) 
   ⎝  ∂r         

In [3]:
# ============================================================
# 0. PARAMETERS
# ============================================================

beta, kappa = sp.symbols("beta kappa")

det_e = sp.simplify(e.det())

# ============================================================
# 1. S_a^{mu nu} = E_a^rho S_rho^{mu nu}
# ============================================================

S_lor = [[[
    sp.simplify(
        sum(e_inv[rho, A] * S_down_up_up[rho][mu][nu] for rho in range(4))
    )
    for nu in range(4)]
    for mu in range(4)]
    for A in range(4)
]

# ============================================================
# 2. Lorentz-covariant density divergence
#
# e^{-1} D_mu(e S_a^{mu nu})
#
# lower Lorentz index:
# D_mu X_a = partial_mu X_a - omega^b_{a mu} X_b
# ============================================================

def div_density_S(A, nu):

    result = 0

    for mu in range(4):

        partial_part = sp.diff(
            det_e * S_lor[A][mu][nu],
            coords[mu]
        )

        spin_part = sum(
            omega[B][A][mu] * det_e * S_lor[B][mu][nu]
            for B in range(4)
        )

        result += partial_part - spin_part

    return sp.simplify(result / det_e)

# ============================================================
# 3. TEGR tensor G_a^nu
#
# IMPORTANT:
# With our S convention the torsion term has PLUS sign:
#
# G_a^nu =
# e^{-1} D_mu(e S_a^{mu nu})
# + E_a^lambda T^rho_{mu lambda} S_rho^{nu mu}
# + 1/4 E_a^nu T
# ============================================================

G = [[
    sp.simplify(
        div_density_S(A, nu)
        +
        sum(
            e_inv[lam, A]
            * T_up_down_down[rho][mu][lam]
            * S_down_up_up[rho][nu][mu]
            for lam in range(4)
            for rho in range(4)
            for mu in range(4)
        )
        +
        sp.Rational(1, 4) * e_inv[nu, A] * Tscalar
    )
    for nu in range(4)]
    for A in range(4)
]

# ============================================================
# 4. T_a and T^a
# ============================================================

T_lor_down = [
    sp.simplify(
        sum(e_inv[mu, A] * V_cov[mu] for mu in range(4))
    )
    for A in range(4)
]

T_lor_up = [
    sp.simplify(
        sum(e[A, mu] * V_contra[mu] for mu in range(4))
    )
    for A in range(4)
]

T2_scalar = sp.simplify(
    sum(V_cov[mu] * V_contra[mu] for mu in range(4))
)

# ============================================================
# 5. A_a^{nu rho}
#
# A_a^{nu rho}
# =
# 2e(T^rho E_a^nu - T^nu E_a^rho)
# ============================================================

A_tensor = [[[
    sp.simplify(
        2 * det_e *
        (
            V_contra[rho] * e_inv[nu, A]
            -
            V_contra[nu] * e_inv[rho, A]
        )
    )
    for rho in range(4)]
    for nu in range(4)]
    for A in range(4)
]

# ============================================================
# 6. D_nu A_a^{nu rho}
# ============================================================

def D_A(A, rho):

    result = 0

    for nu in range(4):

        partial_part = sp.diff(
            A_tensor[A][nu][rho],
            coords[nu]
        )

        spin_part = sum(
            omega[B][A][nu] * A_tensor[B][nu][rho]
            for B in range(4)
        )

        result += partial_part - spin_part

    return sp.simplify(result)

# ============================================================
# 7. Beta-sector tensor V_a^rho
#
# V_a^rho =
# T^2 E_a^rho
# - 2 T^rho T_a
# - 2 T^mu E_a^nu T^rho_{nu mu}
# - e^{-1}D_nu[2e(T^rho E_a^nu - T^nu E_a^rho)]
# ============================================================

V = [[
    sp.simplify(
        T2_scalar * e_inv[rho, A]
        -
        2 * V_contra[rho] * T_lor_down[A]
        -
        2 * sum(
            V_contra[mu]
            * e_inv[nu, A]
            * T_up_down_down[rho][nu][mu]
            for mu in range(4)
            for nu in range(4)
        )
        -
        D_A(A, rho) / det_e
    )
    for rho in range(4)]
    for A in range(4)
]

# ============================================================
# 8. Full LHS
#
# G_a^nu + beta/4 V_a^nu
# ============================================================

LHS = [[
    sp.simplify(
        G[A][nu] + beta * V[A][nu] / 4
    )
    for nu in range(4)]
    for A in range(4)
]

# ============================================================
# 9. FLRW limit
# ============================================================

subs_FLRW = {
    b: 0,
    sp.diff(b, r): 0,
    sp.diff(b, t): 0,
    c: 1,
    sp.diff(c, t): 0
}

G_FLRW = [[
    sp.simplify(sp.factor(G[A][nu].subs(subs_FLRW)))
    for nu in range(4)]
    for A in range(4)
]

V_FLRW = [[
    sp.simplify(sp.factor(V[A][nu].subs(subs_FLRW)))
    for nu in range(4)]
    for A in range(4)
]

# ============================================================
# 10. Print nonzero G
# ============================================================

print("\n============================================================")
print("Nonzero corrected TEGR tensor G_a^nu")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(G[A][nu])
        if expr != 0:
            print(f"G_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

# ============================================================
# 11. Print nonzero V
# ============================================================

print("\n============================================================")
print("Nonzero beta-sector tensor V_a^nu")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(V[A][nu])
        if expr != 0:
            print(f"V_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

# ============================================================
# 12. Print nonzero full LHS
# ============================================================

print("\n============================================================")
print("Nonzero full LHS: G_a^nu + beta/4 V_a^nu")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(LHS[A][nu])
        if expr != 0:
            print(f"LHS_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

# ============================================================
# 13. FLRW output
# ============================================================

print("\n============================================================")
print("FLRW limit of corrected G_a^nu")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(G_FLRW[A][nu])
        if expr != 0:
            print(f"G_FLRW_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

print("\n============================================================")
print("FLRW limit of V_a^nu")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(V_FLRW[A][nu])
        if expr != 0:
            print(f"V_FLRW_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()


Nonzero corrected TEGR tensor G_a^nu
G_0^0 =
⎛           d       ⎞ ⎛    ∂                           d       ⎞
⎜-b(t, r) + ──(a(t))⎟⋅⎜2⋅r⋅──(b(t, r)) + 3⋅b(t, r) - 3⋅──(a(t))⎟
⎝           dt      ⎠ ⎝    ∂r                          dt      ⎠
────────────────────────────────────────────────────────────────
                            2     3                             
                         2⋅a (t)⋅c (t)                          

G_0^1 =
   ⎛           d       ⎞ ⎛    ∂                           d       ⎞         
-r⋅⎜-b(t, r) + ──(a(t))⎟⋅⎜2⋅r⋅──(b(t, r)) + 3⋅b(t, r) - 3⋅──(a(t))⎟⋅b(t, r) 
   ⎝           dt      ⎠ ⎝    ∂r                          dt      ⎠         
────────────────────────────────────────────────────────────────────────────
                                  3     3                                   
                               2⋅a (t)⋅c (t)                                

G_1^1 =
                                                                      2        ↪
   

In [ ]:
import sympy as sp

# ============================================================
# 0. Coordinates and functions
# ============================================================

t, r, th, ph = sp.symbols("t r theta phi", real=True)
coords = [t, r, th, ph]

a = sp.Function("a")(t)
c = sp.Function("c")(t)
b = sp.Function("b")(t, r)

beta, kappa = sp.symbols("beta kappa")

adot = sp.diff(a, t)
br = sp.diff(b, r)

eta = sp.diag(-1, 1, 1, 1)

# ============================================================
# 1. Radial tetrad e^a_mu
# ============================================================

e = sp.zeros(4, 4)

e[0, 0] = c
e[1, 0] = r*b
e[1, 1] = a
e[2, 2] = r*a
e[3, 3] = r*a*sp.sin(th)

e_inv = sp.simplify(e.inv())

det_e = sp.simplify(e.det())

# ============================================================
# 2. Metric
# ============================================================

g = sp.simplify(e.T * eta * e)
g_inv = sp.simplify(g.inv())

# ============================================================
# 3. Spin connection omega^a_{b mu}
# ============================================================

omega = [[[
    sp.Integer(0)
    for mu in range(4)]
    for B in range(4)]
    for A in range(4)
]

omega[1][2][2] = -1
omega[2][1][2] = 1

omega[1][3][3] = -sp.sin(th)
omega[3][1][3] = sp.sin(th)

omega[2][3][3] = -sp.cos(th)
omega[3][2][3] = sp.cos(th)

# ============================================================
# 4. Lorentz torsion T^a_{mu nu}
# ============================================================

T_lor = [[[
    sp.simplify(
        sp.diff(e[A, nu], coords[mu])
        - sp.diff(e[A, mu], coords[nu])
        + sum(omega[A][B][mu] * e[B, nu] for B in range(4))
        - sum(omega[A][B][nu] * e[B, mu] for B in range(4))
    )
    for nu in range(4)]
    for mu in range(4)]
    for A in range(4)
]

# ============================================================
# 5. Coordinate torsion T^rho_{mu nu}
# ============================================================

T_up_down_down = [[[
    sp.simplify(
        sum(e_inv[rho, A] * T_lor[A][mu][nu] for A in range(4))
    )
    for nu in range(4)]
    for mu in range(4)]
    for rho in range(4)
]

# ============================================================
# 6. Fully lowered torsion T_{rho mu nu}
# ============================================================

T_down_down_down = [[[
    sp.simplify(
        sum(g[rho, sig] * T_up_down_down[sig][mu][nu] for sig in range(4))
    )
    for nu in range(4)]
    for mu in range(4)]
    for rho in range(4)
]

# ============================================================
# 7. Fully raised torsion T^{rho mu nu}
# ============================================================

T_up_up_up = [[[
    sp.simplify(
        sum(
            g_inv[rho, sig]
            * g_inv[mu, alpha]
            * g_inv[nu, beta_i]
            * T_down_down_down[sig][alpha][beta_i]
            for sig in range(4)
            for alpha in range(4)
            for beta_i in range(4)
        )
    )
    for nu in range(4)]
    for mu in range(4)]
    for rho in range(4)
]

# ============================================================
# 8. Torsion vector V_mu = T^rho_{rho mu}
# ============================================================

V_cov = [
    sp.simplify(
        sum(T_up_down_down[rho][rho][mu] for rho in range(4))
    )
    for mu in range(4)
]

V_contra = [
    sp.simplify(
        sum(g_inv[mu, nu] * V_cov[nu] for nu in range(4))
    )
    for mu in range(4)
]

# ============================================================
# 9. Torsion scalar from quadratic definition
# ============================================================

term1 = sp.simplify(
    sp.Rational(1, 4)
    * sum(
        T_up_up_up[rho][mu][nu]
        * T_down_down_down[rho][mu][nu]
        for rho in range(4)
        for mu in range(4)
        for nu in range(4)
    )
)

term2 = sp.simplify(
    sp.Rational(1, 2)
    * sum(
        T_up_up_up[rho][mu][nu]
        * T_down_down_down[nu][mu][rho]
        for rho in range(4)
        for mu in range(4)
        for nu in range(4)
    )
)

term3 = sp.simplify(
    -sum(V_cov[mu] * V_contra[mu] for mu in range(4))
)

Tscalar = sp.simplify(sp.factor(term1 + term2 + term3))

# ============================================================
# 10. Mixed torsion versions for K and S
# ============================================================

T_up_up_down = [[[
    sp.simplify(
        sum(
            g_inv[mu, alpha]
            * g_inv[nu, beta_i]
            * T_down_down_down[alpha][beta_i][rho]
            for alpha in range(4)
            for beta_i in range(4)
        )
    )
    for rho in range(4)]
    for nu in range(4)]
    for mu in range(4)
]

T_down_up_up = [[[
    sp.simplify(
        sum(
            g_inv[mu, alpha]
            * g_inv[nu, beta_i]
            * T_down_down_down[rho][alpha][beta_i]
            for alpha in range(4)
            for beta_i in range(4)
        )
    )
    for nu in range(4)]
    for mu in range(4)]
    for rho in range(4)
]

# ============================================================
# 11. Contorsion K^{mu nu}_{rho}
# ============================================================

K_up_up_down = [[[
    sp.simplify(
        -sp.Rational(1, 2)
        * (
            T_up_up_down[mu][nu][rho]
            - T_up_up_down[nu][mu][rho]
            - T_down_up_up[rho][mu][nu]
        )
    )
    for rho in range(4)]
    for nu in range(4)]
    for mu in range(4)
]

# ============================================================
# 12. Q^nu = T^{alpha nu}_{alpha}
# ============================================================

Q_contra = [
    sp.simplify(
        sum(T_up_up_down[alpha][nu][alpha] for alpha in range(4))
    )
    for nu in range(4)
]

# ============================================================
# 13. Superpotential S_rho^{mu nu}
# ============================================================

S_down_up_up = [[[
    sp.simplify(
        sp.Rational(1, 2)
        * (
            K_up_up_down[mu][nu][rho]
            + (1 if mu == rho else 0) * Q_contra[nu]
            - (1 if nu == rho else 0) * Q_contra[mu]
        )
    )
    for nu in range(4)]
    for mu in range(4)]
    for rho in range(4)
]

# ============================================================
# 14. Check T = S*T
# ============================================================

T_from_S = sp.simplify(
    sp.factor(
        sum(
            S_down_up_up[rho][mu][nu]
            * T_up_down_down[rho][mu][nu]
            for rho in range(4)
            for mu in range(4)
            for nu in range(4)
        )
    )
)

check_T_from_S = sp.simplify(sp.factor(T_from_S - Tscalar))

# ============================================================
# 15. S_a^{mu nu} = E_a^rho S_rho^{mu nu}
# ============================================================

S_lor = [[[
    sp.simplify(
        sum(e_inv[rho, A] * S_down_up_up[rho][mu][nu] for rho in range(4))
    )
    for nu in range(4)]
    for mu in range(4)]
    for A in range(4)
]

# ============================================================
# 16. Lorentz-covariant density divergence
# e^{-1} D_mu(e S_a^{mu nu})
# lower Lorentz index: D_mu X_a = partial_mu X_a - omega^b_{a mu} X_b
# ============================================================

def div_density_S(A, nu):
    result = 0

    for mu in range(4):
        partial_part = sp.diff(det_e * S_lor[A][mu][nu], coords[mu])

        spin_part = sum(
            omega[B][A][mu] * det_e * S_lor[B][mu][nu]
            for B in range(4)
        )

        result += partial_part - spin_part

    return sp.simplify(result / det_e)

# ============================================================
# 17. Teleparallel TEGR tensor candidate
# G_a^nu =
# e^{-1}D_mu(e S_a^{mu nu})
# + E_a^lambda T^rho_{mu lambda} S_rho^{nu mu}
# + 1/4 E_a^nu T
# ============================================================

G_TP = [[
    sp.simplify(
        div_density_S(A, nu)
        +
        sum(
            e_inv[lam, A]
            * T_up_down_down[rho][mu][lam]
            * S_down_up_up[rho][nu][mu]
            for lam in range(4)
            for rho in range(4)
            for mu in range(4)
        )
        +
        sp.Rational(1, 4) * e_inv[nu, A] * Tscalar
    )
    for nu in range(4)]
    for A in range(4)
]

# ============================================================
# 18. Levi-Civita Christoffel symbols
# ============================================================

Gamma = [[[
    sp.Integer(0)
    for nu in range(4)]
    for mu in range(4)]
    for lam in range(4)
]

for lam in range(4):
    for mu in range(4):
        for nu in range(4):
            expr = 0
            for sig in range(4):
                expr += sp.Rational(1, 2) * g_inv[lam, sig] * (
                    sp.diff(g[sig, mu], coords[nu])
                    + sp.diff(g[sig, nu], coords[mu])
                    - sp.diff(g[mu, nu], coords[sig])
                )
            Gamma[lam][mu][nu] = sp.simplify(expr)

# ============================================================
# 19. Ricci tensor R_{mu nu}
# R_{mu nu} =
# partial_lam Gamma^lam_{mu nu}
# - partial_nu Gamma^lam_{mu lam}
# + Gamma^lam_{mu nu} Gamma^rho_{lam rho}
# - Gamma^rho_{mu lam} Gamma^lam_{nu rho}
# ============================================================

R_mu_nu = [[sp.Integer(0) for nu in range(4)] for mu in range(4)]

for mu in range(4):
    for nu in range(4):
        expr = 0

        for lam in range(4):
            expr += sp.diff(Gamma[lam][mu][nu], coords[lam])
            expr -= sp.diff(Gamma[lam][mu][lam], coords[nu])

            for rho in range(4):
                expr += Gamma[lam][mu][nu] * Gamma[rho][lam][rho]
                expr -= Gamma[rho][mu][lam] * Gamma[lam][nu][rho]

        R_mu_nu[mu][nu] = sp.simplify(expr)

# ============================================================
# 20. Ricci scalar and LC Einstein tensor
# ============================================================

R_scalar = sp.simplify(
    sum(
        g_inv[mu, nu] * R_mu_nu[mu][nu]
        for mu in range(4)
        for nu in range(4)
    )
)

G_LC_down_down = [[
    sp.simplify(
        R_mu_nu[mu][nu]
        - sp.Rational(1, 2) * g[mu, nu] * R_scalar
    )
    for nu in range(4)]
    for mu in range(4)
]

G_LC_mixed = [[
    sp.simplify(
        sum(g_inv[nu, sig] * G_LC_down_down[mu][sig] for sig in range(4))
    )
    for nu in range(4)]
    for mu in range(4)
]

# ============================================================
# 21. LC tetrad projection with 1/2 normalization
# G_LC_a^nu = 1/2 E_a^mu G_mu^nu
# ============================================================

G_LC_proj = [[
    sp.simplify(
        sp.Rational(1, 2)
        * sum(e_inv[mu, A] * G_LC_mixed[mu][nu] for mu in range(4))
    )
    for nu in range(4)]
    for A in range(4)
]

# ============================================================
# 22. FLRW substitution
# ============================================================

subs_FLRW = {
    b: 0,
    sp.diff(b, r): 0,
    sp.diff(b, t): 0,
    sp.diff(b, (r, 2)): 0,
    sp.diff(b, t, r): 0,
    c: 1,
    sp.diff(c, t): 0,
    sp.diff(c, (t, 2)): 0,
}

G_TP_FLRW = [[
    sp.simplify(sp.factor(G_TP[A][nu].subs(subs_FLRW)))
    for nu in range(4)]
    for A in range(4)
]

G_LC_FLRW = [[
    sp.simplify(sp.factor(G_LC_proj[A][nu].subs(subs_FLRW)))
    for nu in range(4)]
    for A in range(4)
]

Diff_FLRW = [[
    sp.simplify(sp.factor(G_TP_FLRW[A][nu] - G_LC_FLRW[A][nu]))
    for nu in range(4)]
    for A in range(4)
]

# ============================================================
# 23. Optional: beta-sector tensor V_a^nu
# ============================================================

T_lor_down = [
    sp.simplify(
        sum(e_inv[mu, A] * V_cov[mu] for mu in range(4))
    )
    for A in range(4)
]

T2_scalar = sp.simplify(
    sum(V_cov[mu] * V_contra[mu] for mu in range(4))
)

A_tensor = [[[
    sp.simplify(
        2 * det_e *
        (
            V_contra[rho] * e_inv[nu, A]
            -
            V_contra[nu] * e_inv[rho, A]
        )
    )
    for rho in range(4)]
    for nu in range(4)]
    for A in range(4)
]

def D_A(A, rho):
    result = 0

    for nu in range(4):
        partial_part = sp.diff(A_tensor[A][nu][rho], coords[nu])

        spin_part = sum(
            omega[B][A][nu] * A_tensor[B][nu][rho]
            for B in range(4)
        )

        result += partial_part - spin_part

    return sp.simplify(result)

V_beta = [[
    sp.simplify(
        T2_scalar * e_inv[rho, A]
        -
        2 * V_contra[rho] * T_lor_down[A]
        -
        2 * sum(
            V_contra[mu]
            * e_inv[nu, A]
            * T_up_down_down[rho][nu][mu]
            for mu in range(4)
            for nu in range(4)
        )
        -
        D_A(A, rho) / det_e
    )
    for rho in range(4)]
    for A in range(4)
]

V_beta_FLRW = [[
    sp.simplify(sp.factor(V_beta[A][nu].subs(subs_FLRW)))
    for nu in range(4)]
    for A in range(4)
]

# ============================================================
# 24. Print checks
# ============================================================

print("\n============================================================")
print("CHECK 1: T_from_S - Tscalar")
print("============================================================")
sp.pprint(check_T_from_S)

print("\n============================================================")
print("Torsion scalar T")
print("============================================================")
sp.pprint(sp.factor(Tscalar))

print("\n============================================================")
print("FLRW limit of torsion scalar")
print("============================================================")
sp.pprint(sp.factor(Tscalar.subs(subs_FLRW)))

print("\n============================================================")
print("FLRW: Teleparallel G_TP_a^nu")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(G_TP_FLRW[A][nu])
        if expr != 0:
            print(f"G_TP_FLRW_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

print("\n============================================================")
print("FLRW: Levi-Civita projected 1/2 G_LC_a^nu")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(G_LC_FLRW[A][nu])
        if expr != 0:
            print(f"G_LC_FLRW_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

print("\n============================================================")
print("FLRW difference: G_TP - 1/2 G_LC")
print("============================================================")

all_zero = True

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(Diff_FLRW[A][nu])
        if expr != 0:
            all_zero = False
            print(f"Delta_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

if all_zero:
    print("All differences are zero.")

print("\n============================================================")
print("FLRW: beta-sector V_a^nu")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(V_beta_FLRW[A][nu])
        if expr != 0:
            print(f"V_FLRW_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

            # ============================================================
# DEBUG: Split teleparallel G into A + B + C
# ============================================================

A_part = [[
    sp.simplify(div_density_S(A, nu))
    for nu in range(4)]
    for A in range(4)
]

B_part = [[
    sp.simplify(
        sum(
            e_inv[lam, A]
            * T_up_down_down[rho][mu][lam]
            * S_down_up_up[rho][nu][mu]
            for lam in range(4)
            for rho in range(4)
            for mu in range(4)
        )
    )
    for nu in range(4)]
    for A in range(4)
]

C_part = [[
    sp.simplify(
        sp.Rational(1, 4) * e_inv[nu, A] * Tscalar
    )
    for nu in range(4)]
    for A in range(4)
]

A_FLRW = [[sp.simplify(sp.factor(A_part[A][nu].subs(subs_FLRW))) for nu in range(4)] for A in range(4)]
B_FLRW = [[sp.simplify(sp.factor(B_part[A][nu].subs(subs_FLRW))) for nu in range(4)] for A in range(4)]
C_FLRW = [[sp.simplify(sp.factor(C_part[A][nu].subs(subs_FLRW))) for nu in range(4)] for A in range(4)]

print("\n============================================================")
print("FLRW A_part = e^{-1}D_mu(e S_a^{mu nu})")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(A_FLRW[A][nu])
        if expr != 0:
            print(f"A_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

print("\n============================================================")
print("FLRW B_part = E_a^lambda T^rho_{mu lambda} S_rho^{nu mu}")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(B_FLRW[A][nu])
        if expr != 0:
            print(f"B_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

print("\n============================================================")
print("FLRW C_part = 1/4 E_a^nu T")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(C_FLRW[A][nu])
        if expr != 0:
            print(f"C_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

print("\n============================================================")
print("FLRW A + B + C")
print("============================================================")

ABC_FLRW = [[
    sp.simplify(sp.factor(A_FLRW[A][nu] + B_FLRW[A][nu] + C_FLRW[A][nu]))
    for nu in range(4)]
    for A in range(4)
]

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(ABC_FLRW[A][nu])
        if expr != 0:
            print(f"ABC_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

print("\n============================================================")
print("FLRW LC projection")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(G_LC_FLRW[A][nu])
        if expr != 0:
            print(f"LC_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

print("\n============================================================")
print("FLRW difference by component: A+B+C - LC")
print("============================================================")

for A in range(4):
    for nu in range(4):
        expr = sp.simplify(sp.factor(ABC_FLRW[A][nu] - G_LC_FLRW[A][nu]))
        if expr != 0:
            print(f"Delta_{A}^{nu} =")
            sp.pprint(sp.factor(expr))
            print()

# ============================================================
# Try alternate sign/index variants for B term
# ============================================================

B1 = B_part

B2 = [[
    sp.simplify(-B_part[A][nu])
    for nu in range(4)]
    for A in range(4)
]

B3 = [[
    sp.simplify(
        sum(
            e_inv[lam, A]
            * T_up_down_down[rho][lam][mu]
            * S_down_up_up[rho][nu][mu]
            for lam in range(4)
            for rho in range(4)
            for mu in range(4)
        )
    )
    for nu in range(4)]
    for A in range(4)
]

B4 = [[
    sp.simplify(-B3[A][nu])
    for nu in range(4)]
    for A in range(4)
]

B5 = [[
    sp.simplify(
        sum(
            e_inv[lam, A]
            * T_up_down_down[rho][mu][lam]
            * S_down_up_up[rho][mu][nu]
            for lam in range(4)
            for rho in range(4)
            for mu in range(4)
        )
    )
    for nu in range(4)]
    for A in range(4)
]

B6 = [[
    sp.simplify(-B5[A][nu])
    for nu in range(4)]
    for A in range(4)
]

B_variants = {
    "B1 = + T[rho][mu][lam] S[rho][nu][mu]": B1,
    "B2 = - T[rho][mu][lam] S[rho][nu][mu]": B2,
    "B3 = + T[rho][lam][mu] S[rho][nu][mu]": B3,
    "B4 = - T[rho][lam][mu] S[rho][nu][mu]": B4,
    "B5 = + T[rho][mu][lam] S[rho][mu][nu]": B5,
    "B6 = - T[rho][mu][lam] S[rho][mu][nu]": B6,
}

print("\n============================================================")
print("TEST B variants against LC in FLRW")
print("============================================================")

for name, Bv in B_variants.items():

    Bv_FLRW = [[
        sp.simplify(sp.factor(Bv[A][nu].subs(subs_FLRW)))
        for nu in range(4)]
        for A in range(4)
    ]

    Gv_FLRW = [[
        sp.simplify(sp.factor(A_FLRW[A][nu] + Bv_FLRW[A][nu] + C_FLRW[A][nu]))
        for nu in range(4)]
        for A in range(4)
    ]

    Diff_v = [[
        sp.simplify(sp.factor(Gv_FLRW[A][nu] - G_LC_FLRW[A][nu]))
        for nu in range(4)]
        for A in range(4)
    ]

    ok = True
    for A in range(4):
        for nu in range(4):
            if Diff_v[A][nu] != 0:
                ok = False

    print("\n", name)
    if ok:
        print("MATCHES LC: all differences zero")
    else:
        print("Not matching. Nonzero differences:")
        for A in range(4):
            for nu in range(4):
                expr = sp.simplify(Diff_v[A][nu])
                if expr != 0:
                    print(f"Delta_{A}^{nu} =")
                    sp.pprint(sp.factor(expr))
                    print()


print("\n============================================================")
print("DONE")
print("============================================================")

In [4]:
# ============================================================
# MATTER SECTOR — COVARIANT STRUCTURE
#
# Coordinates: (t, r, theta, phi)
#
# Field equation:
#
# LHS_a^nu = G_a^nu + beta/4 V_a^nu
# RHS_a^nu = kappa/2 Theta_a^nu
#
# Full equation:
#
# LHS_a^nu - RHS_a^nu = 0
# ============================================================

rho = sp.Function("rho")(t, r)
p = sp.Function("p")(t, r)
W = sp.Function("W")(t, r)

# ============================================================
# 1. GENERAL FLUID 4-VELOCITY
#
# We keep u^mu general in the radial sector:
#
# u^mu = (u0, ur, 0, 0)
# ============================================================

u0 = sp.Function("u0")(t, r)
ur = sp.Function("ur")(t, r)

u_contra = sp.Matrix([u0, ur, 0, 0])

# u_mu = g_{mu nu} u^nu
u_cov = sp.simplify(g * u_contra)

# tetrad-frame velocity:
# u^a = e^a_mu u^mu
u_lor_contra = sp.simplify(e * u_contra)

# u_a = eta_ab u^b
eta = sp.diag(-1, 1, 1, 1)
u_lor_cov = sp.simplify(eta * u_lor_contra)

# normalization check:
u_norm_coord = sp.simplify((u_contra.T * g * u_contra)[0])
u_norm_lor = sp.simplify((u_lor_contra.T * eta * u_lor_contra)[0])

# ============================================================
# 2. PERFECT FLUID IN COORDINATE BASIS
#
# Convention:
#
# Theta^{mu nu} = (rho + p) u^mu u^nu + p g^{mu nu}
#
# If rho already means energy density, use rho.
# If rho means mass density, replace rho by rho*c**2.
# ============================================================

Theta_contra_PF = sp.zeros(4, 4)

g_inv = sp.simplify(g.inv())

for mu in range(4):
    for nu in range(4):
        Theta_contra_PF[mu, nu] = sp.simplify(
            (rho + p) * u_contra[mu] * u_contra[nu]
            + p * g_inv[mu, nu]
        )

# ============================================================
# 3. OPTIONAL: RADIAL EFFECTIVE MATTER ANSATZ
#
# This is your older ansatz:
#
# Theta^{mu nu} =
# [[rho, rho W, 0, 0],
#  [rho W, p, 0, 0],
#  [0, 0, p/r^2, 0],
#  [0, 0, 0, p/(r^2 sin^2 theta)]]
#
# Use this if you want the matter flow W to be imposed directly.
# ============================================================

Theta_contra_ansatz = sp.zeros(4, 4)

Theta_contra_ansatz[0, 0] = rho
Theta_contra_ansatz[0, 1] = rho * W
Theta_contra_ansatz[1, 0] = rho * W
Theta_contra_ansatz[1, 1] = p
Theta_contra_ansatz[2, 2] = p / r**2
Theta_contra_ansatz[3, 3] = p / (r**2 * sp.sin(th)**2)

# ============================================================
# Choose matter model here:
#
# option 1: perfect fluid from u^mu
# option 2: direct radial ansatz
# ============================================================

use_perfect_fluid = True

if use_perfect_fluid:
    Theta_contra = Theta_contra_PF
else:
    Theta_contra = Theta_contra_ansatz

# ============================================================
# 4. INDEX CONVERSIONS
#
# Theta_mu^nu = g_mu alpha Theta^{alpha nu}
#
# Theta_a^nu = e_a^mu Theta_mu^nu
#
# Theta_a^b = Theta_a^nu e^b_nu
# ============================================================

Theta_mixed_coord = sp.simplify(g * Theta_contra)

Theta_lor_coord = sp.simplify(e_inv * Theta_mixed_coord)

Theta_lor_lor = sp.simplify(Theta_lor_coord * e.T)

# ============================================================
# 5. SPECIAL COMOVING-TETRAD CHOICE
#
# u^a = (1, 0, 0, 0)
#
# Then:
#
# u^mu = e_a^mu u^a
# u_mu = g_mu nu u^nu
#
# This is the important non-rigid version.
# ============================================================

u_lor_comoving = sp.Matrix([1, 0, 0, 0])

u_coord_from_lor = sp.simplify(e_inv.T * u_lor_comoving)

u_cov_from_lor = sp.simplify(g * u_coord_from_lor)

print("\n============================================================")
print("COMOVING TETRAD VELOCITY")
print("u^a = (1,0,0,0)")
print("u^mu = e_a^mu u^a")
print("u_mu = g_mu nu u^nu")
print("============================================================")

print("\nu^mu:")
sp.pprint(u_coord_from_lor)

print("\nu_mu:")
sp.pprint(u_cov_from_lor)

# ============================================================
# 6. LOCAL RADIAL BOOST
#
# Boost in the local Lorentz 0-1 plane:
#
# Lambda^a_b =
# [[gamma, gamma v, 0, 0],
#  [gamma v, gamma, 0, 0],
#  [0, 0, 1, 0],
#  [0, 0, 0, 1]]
#
# beta_v is local velocity parameter.
# ============================================================

vloc = sp.Function("vloc")(t, r)
gamma = sp.Function("gamma")(t, r)

Lambda = sp.Matrix([
    [gamma, gamma * vloc, 0, 0],
    [gamma * vloc, gamma, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
])

Lambda_inv = sp.Matrix([
    [gamma, -gamma * vloc, 0, 0],
    [-gamma * vloc, gamma, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1],
])

# impose gamma^2 (1 - v^2) = 1 only later if desired
boost_relation = sp.Eq(gamma**2 * (1 - vloc**2), 1)

# boosted tetrad:
#
# ebar^a_mu = Lambda^a_b e^b_mu
#
# inverse:
#
# ebar_a^mu = Lambda_a^b e_b^mu
# approximately represented by Lambda_inv acting on Lorentz index.
# ============================================================

e_boost = sp.simplify(Lambda * e)

e_inv_boost = sp.simplify(e_inv * Lambda_inv)

# ============================================================
# 7. SPIN CONNECTION FROM LOCAL LORENTZ BOOST
#
# omega^a_{b mu} = Lambda^a_c partial_mu (Lambda^{-1})^c_b
#
# This separates inertial spin-connection contributions.
# ============================================================

coords = [t, r, th, ph]

omega_boost = {}

for mu_index, coord in enumerate(coords):
    omega_mu = sp.simplify(Lambda * sp.diff(Lambda_inv, coord))
    omega_boost[mu_index] = omega_mu

print("\n============================================================")
print("BOOSTED TETRAD ebar^a_mu")
print("============================================================")
sp.pprint(e_boost)

print("\n============================================================")
print("BOOST SPIN CONNECTION omega^a_{b mu}")
print("omega_mu = Lambda d_mu Lambda^{-1}")
print("============================================================")

for mu_index, coord in enumerate(coords):
    print(f"\nomega_mu for coordinate {coord}:")
    sp.pprint(omega_boost[mu_index])

# ============================================================
# 8. FIELD EQUATIONS WITH LHS AND RHS SEPARATED
#
# LHS_a^nu = G_a^nu + beta/4 V_a^nu
# RHS_a^nu = kappa/2 Theta_a^nu
# ============================================================

FieldLHS = [[
    sp.simplify(sp.factor(LHS[A][nu]))
    for nu in range(4)]
    for A in range(4)
]

FieldRHS = [[
    sp.simplify(sp.factor(sp.Rational(1, 2) * kappa * Theta_lor_coord[A, nu]))
    for nu in range(4)]
    for A in range(4)
]

FieldEq = [[
    sp.simplify(sp.factor(FieldLHS[A][nu] - FieldRHS[A][nu]))
    for nu in range(4)]
    for A in range(4)
]

# ============================================================
# 9. PRINT MATTER TENSORS
# ============================================================

print("\n============================================================")
print("u^mu")
print("============================================================")
sp.pprint(u_contra)

print("\n============================================================")
print("u_mu = g_mu nu u^nu")
print("============================================================")
sp.pprint(u_cov)

print("\n============================================================")
print("u^a = e^a_mu u^mu")
print("============================================================")
sp.pprint(u_lor_contra)

print("\n============================================================")
print("u_a = eta_ab u^b")
print("============================================================")
sp.pprint(u_lor_cov)

print("\n============================================================")
print("Normalization check")
print("u^mu g_mu nu u^nu")
print("============================================================")
sp.pprint(u_norm_coord)

print("\n============================================================")
print("Theta^{mu nu}")
print("============================================================")
sp.pprint(Theta_contra)

print("\n============================================================")
print("Theta_mu^nu = g_mu alpha Theta^{alpha nu}")
print("============================================================")
sp.pprint(Theta_mixed_coord)

print("\n============================================================")
print("Theta_a^nu = e_a^mu Theta_mu^nu")
print("============================================================")
sp.pprint(Theta_lor_coord)

print("\n============================================================")
print("Theta_a^b = Theta_a^nu e^b_nu")
print("============================================================")
sp.pprint(Theta_lor_lor)

# ============================================================
# 10. PRINT FIELD EQUATIONS AS LHS = RHS
# ============================================================

print("\n============================================================")
print("FULL FIELD EQUATIONS")
print("LHS_a^nu = RHS_a^nu")
print("where LHS = G_a^nu + beta/4 V_a^nu")
print("and   RHS = kappa/2 Theta_a^nu")
print("============================================================")

for A in range(4):
    for nu in range(4):

        lhs_expr = sp.simplify(FieldLHS[A][nu])
        rhs_expr = sp.simplify(FieldRHS[A][nu])
        eq_expr = sp.simplify(FieldEq[A][nu])

        if lhs_expr != 0 or rhs_expr != 0:

            print(f"\nField equation ({A},{nu}):")
            print("LHS:")
            sp.pprint(lhs_expr)
            print("\nRHS:")
            sp.pprint(rhs_expr)
            print("\nLHS - RHS = 0:")
            sp.pprint(eq_expr)
            print()

# ============================================================
# 11. SELECTED FIELD EQUATIONS
# ============================================================

independent_indices = [
    (0, 0),
    (0, 1),
    (1, 0),
    (1, 1),
    (2, 2),
    (3, 3),
]

print("\n============================================================")
print("SELECTED FIELD EQUATIONS")
print("============================================================")

for A, nu in independent_indices:

    print(f"\nField equation ({A},{nu}):")
    print("LHS:")
    sp.pprint(FieldLHS[A][nu])

    print("\nRHS:")
    sp.pprint(FieldRHS[A][nu])

    print("\nLHS - RHS = 0:")
    sp.pprint(FieldEq[A][nu])
    print()

# ============================================================
# 12. ANGULAR CONSISTENCY CHECK
#
# Because e^3_phi has an extra sin(theta):
#
# FieldEq_3^3 * sin(theta) - FieldEq_2^2
# ============================================================

angular_check = sp.simplify(
    sp.factor(
        FieldEq[3][3] * sp.sin(th)
        - FieldEq[2][2]
    )
)

print("\n============================================================")
print("Angular consistency check")
print("FieldEq_3^3 * sin(theta) - FieldEq_2^2")
print("============================================================")
sp.pprint(angular_check)

# ============================================================
# 13. FLRW MATTER LIMIT CHECK
#
# b=0, c=1, W=0,
# rho=rho(t), p=p(t),
# u^mu=(1,0,0,0)
# ============================================================

rho_t = sp.Function("rho")(t)
p_t = sp.Function("p")(t)

subs_matter_FLRW = {
    b: 0,
    sp.diff(b, r): 0,
    sp.diff(b, t): 0,
    sp.diff(b, (r, 2)): 0,
    sp.diff(b, t, r): 0,

    c: 1,
    sp.diff(c, t): 0,
    sp.diff(c, (t, 2)): 0,

    W: 0,
    rho: rho_t,
    p: p_t,

    u0: 1,
    ur: 0,
}

FieldLHS_FLRW = [[
    sp.simplify(sp.factor(FieldLHS[A][nu].subs(subs_matter_FLRW)))
    for nu in range(4)]
    for A in range(4)
]

FieldRHS_FLRW = [[
    sp.simplify(sp.factor(FieldRHS[A][nu].subs(subs_matter_FLRW)))
    for nu in range(4)]
    for A in range(4)
]

FieldEq_FLRW = [[
    sp.simplify(sp.factor(FieldEq[A][nu].subs(subs_matter_FLRW)))
    for nu in range(4)]
    for A in range(4)
]

print("\n============================================================")
print("FLRW LIMIT FIELD EQUATIONS")
print("============================================================")

for A in range(4):
    for nu in range(4):

        lhs_expr = sp.simplify(FieldLHS_FLRW[A][nu])
        rhs_expr = sp.simplify(FieldRHS_FLRW[A][nu])
        eq_expr = sp.simplify(FieldEq_FLRW[A][nu])

        if lhs_expr != 0 or rhs_expr != 0:

            print(f"\nFieldEq_FLRW_{A}^{nu}:")
            print("LHS:")
            sp.pprint(lhs_expr)

            print("\nRHS:")
            sp.pprint(rhs_expr)

            print("\nLHS - RHS = 0:")
            sp.pprint(eq_expr)
            print()


COMOVING TETRAD VELOCITY
u^a = (1,0,0,0)
u^mu = e_a^mu u^a
u_mu = g_mu nu u^nu

u^mu:
⎡ 1  ⎤
⎢────⎥
⎢c(t)⎥
⎢    ⎥
⎢ 0  ⎥
⎢    ⎥
⎢ 0  ⎥
⎢    ⎥
⎣ 0  ⎦

u_mu:
⎡ 2  2             ⎤
⎢r ⋅b (t, r)       ⎥
⎢─────────── - c(t)⎥
⎢   c(t)           ⎥
⎢                  ⎥
⎢  r⋅a(t)⋅b(t, r)  ⎥
⎢  ──────────────  ⎥
⎢       c(t)       ⎥
⎢                  ⎥
⎢        0         ⎥
⎢                  ⎥
⎣        0         ⎦

BOOSTED TETRAD ebar^a_mu
⎡(r⋅b(t, r)⋅vloc(t, r) + c(t))⋅γ(t, r)  a(t)⋅γ(t, r)⋅vloc(t, r)    0           ↪
⎢                                                                              ↪
⎢(r⋅b(t, r) + c(t)⋅vloc(t, r))⋅γ(t, r)       a(t)⋅γ(t, r)          0           ↪
⎢                                                                              ↪
⎢                  0                               0             r⋅a(t)        ↪
⎢                                                                              ↪
⎣                  0                               0               0     r⋅a(t 

In [5]:
# ============================================================
# MATTER SECTOR — SIMPLE TETRAD-COMOVING PERFECT FLUID
#
# Field equation:
#
# LHS_a^nu = RHS_a^nu
#
# LHS_a^nu = G_a^nu + beta/4 V_a^nu
# RHS_a^nu = kappa/2 Theta_a^nu
#
# Matter choice:
#
# u^a = (1,0,0,0)
# Theta_a^b = diag(-rho, p, p, p)
# Theta_a^nu = Theta_a^b e_b^nu
# ============================================================

rho = sp.Function("rho")(t, r)
p = sp.Function("p")(t, r)

eta = sp.diag(-1, 1, 1, 1)

# ============================================================
# TETRAD-COMOVING PERFECT FLUID
#
# Mixed Lorentz tensor:
#
# Theta_a^b = diag(-rho, p, p, p)
# ============================================================

Theta_lor_lor = sp.diag(-rho, p, p, p)

# ============================================================
# Convert to Theta_a^nu
#
# Theta_a^nu = Theta_a^b e_b^nu
#
# Here e_inv = e_a^mu
# ============================================================

Theta_lor_coord = sp.simplify(Theta_lor_lor * e_inv)

# ============================================================
# Optional coordinate mixed tensor
#
# Theta_mu^nu = e^a_mu Theta_a^nu
# ============================================================

Theta_mixed_coord = sp.simplify(e.T * Theta_lor_coord)

# ============================================================
# Optional contravariant tensor
#
# Theta^{mu nu} = g^{mu alpha} Theta_alpha^nu
# ============================================================

g_inv = sp.simplify(g.inv())
Theta_contra = sp.simplify(g_inv * Theta_mixed_coord)

# ============================================================
# FIELD EQUATIONS
# ============================================================

FieldLHS = [[
    sp.simplify(sp.factor(LHS[A][nu]))
    for nu in range(4)]
    for A in range(4)
]

FieldRHS = [[
    sp.simplify(
        sp.factor(sp.Rational(1, 2) * kappa * Theta_lor_coord[A, nu])
    )
    for nu in range(4)]
    for A in range(4)
]

FieldEq = [[
    sp.simplify(sp.factor(FieldLHS[A][nu] - FieldRHS[A][nu]))
    for nu in range(4)]
    for A in range(4)
]

# ============================================================
# PRINT MATTER TENSORS
# ============================================================

print("\n============================================================")
print("Theta_a^b = diag(-rho, p, p, p)")
print("============================================================")
sp.pprint(Theta_lor_lor)

print("\n============================================================")
print("Theta_a^nu = Theta_a^b e_b^nu")
print("============================================================")
sp.pprint(Theta_lor_coord)

print("\n============================================================")
print("Theta_mu^nu = e^a_mu Theta_a^nu")
print("============================================================")
sp.pprint(Theta_mixed_coord)

# ============================================================
# PRINT FIELD EQUATIONS AS LHS = RHS
# ============================================================

print("\n============================================================")
print("FULL FIELD EQUATIONS")
print("LHS_a^nu = RHS_a^nu")
print("============================================================")

for A in range(4):
    for nu in range(4):

        lhs_expr = sp.simplify(FieldLHS[A][nu])
        rhs_expr = sp.simplify(FieldRHS[A][nu])
        eq_expr = sp.simplify(FieldEq[A][nu])

        if lhs_expr != 0 or rhs_expr != 0:

            print(f"\nField equation ({A},{nu}):")

            print("LHS:")
            sp.pprint(lhs_expr)

            print("\nRHS:")
            sp.pprint(rhs_expr)

            print("\nLHS - RHS = 0:")
            sp.pprint(eq_expr)
            print()

# ============================================================
# SELECTED FIELD EQUATIONS
# ============================================================

independent_indices = [
    (0, 0),
    (0, 1),
    (1, 0),
    (1, 1),
    (2, 2),
    (3, 3),
]

print("\n============================================================")
print("SELECTED FIELD EQUATIONS")
print("============================================================")

for A, nu in independent_indices:

    print(f"\nField equation ({A},{nu}):")

    print("LHS:")
    sp.pprint(FieldLHS[A][nu])

    print("\nRHS:")
    sp.pprint(FieldRHS[A][nu])

    print("\nLHS - RHS = 0:")
    sp.pprint(FieldEq[A][nu])
    print()

# ============================================================
# ANGULAR CONSISTENCY CHECK
# ============================================================

angular_check = sp.simplify(
    sp.factor(
        FieldEq[3][3] * sp.sin(th)
        - FieldEq[2][2]
    )
)

print("\n============================================================")
print("Angular consistency check")
print("FieldEq_3^3 * sin(theta) - FieldEq_2^2")
print("============================================================")
sp.pprint(angular_check)

# ============================================================
# FLRW LIMIT
# ============================================================

rho_t = sp.Function("rho")(t)
p_t = sp.Function("p")(t)

subs_matter_FLRW = {
    b: 0,
    sp.diff(b, r): 0,
    sp.diff(b, t): 0,
    sp.diff(b, (r, 2)): 0,
    sp.diff(b, t, r): 0,

    c: 1,
    sp.diff(c, t): 0,
    sp.diff(c, (t, 2)): 0,

    rho: rho_t,
    p: p_t,
}

FieldLHS_FLRW = [[
    sp.simplify(sp.factor(FieldLHS[A][nu].subs(subs_matter_FLRW)))
    for nu in range(4)]
    for A in range(4)
]

FieldRHS_FLRW = [[
    sp.simplify(sp.factor(FieldRHS[A][nu].subs(subs_matter_FLRW)))
    for nu in range(4)]
    for A in range(4)
]

FieldEq_FLRW = [[
    sp.simplify(sp.factor(FieldEq[A][nu].subs(subs_matter_FLRW)))
    for nu in range(4)]
    for A in range(4)
]

print("\n============================================================")
print("FLRW LIMIT FIELD EQUATIONS")
print("============================================================")

for A in range(4):
    for nu in range(4):

        lhs_expr = FieldLHS_FLRW[A][nu]
        rhs_expr = FieldRHS_FLRW[A][nu]
        eq_expr = FieldEq_FLRW[A][nu]

        if lhs_expr != 0 or rhs_expr != 0:

            print(f"\nFieldEq_FLRW_{A}^{nu}:")

            print("LHS:")
            sp.pprint(lhs_expr)

            print("\nRHS:")
            sp.pprint(rhs_expr)

            print("\nLHS - RHS = 0:")
            sp.pprint(eq_expr)
            print()


Theta_a^b = diag(-rho, p, p, p)
⎡-ρ(t, r)     0        0        0   ⎤
⎢                                   ⎥
⎢   0      p(t, r)     0        0   ⎥
⎢                                   ⎥
⎢   0         0     p(t, r)     0   ⎥
⎢                                   ⎥
⎣   0         0        0     p(t, r)⎦

Theta_a^nu = Theta_a^b e_b^nu
⎡     -ρ(t, r)                                       ⎤
⎢     ─────────          0        0           0      ⎥
⎢       c(t)                                         ⎥
⎢                                                    ⎥
⎢-r⋅b(t, r)⋅p(t, r)   p(t, r)                        ⎥
⎢───────────────────  ───────     0           0      ⎥
⎢     a(t)⋅c(t)        a(t)                          ⎥
⎢                                                    ⎥
⎢                              p(t, r)               ⎥
⎢         0              0     ───────        0      ⎥
⎢                              r⋅a(t)                ⎥
⎢                                                    ⎥
⎢         